# Episode 5 — Pytrees and SGD

**Instructor notebook** · run top-to-bottom before recording.

A 2-layer MLP as a nested **dict** of params; update every leaf with `tree_map` and a fixed learning rate.

| | |
|---|---|
| **Chapter** | 1.5 · Part I — Pure JAX |
| **Prereq** | Episodes 1–4 |
| **Next** | Part II — GPT-2 transformer |

**JAX docs:** [Pytrees](https://docs.jax.dev/en/latest/pytrees.html) · [`jax.tree.map`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.map.html) · [`jax.tree.leaves`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.leaves.html)


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr

## Pytrees — nested dicts of arrays

JAX walks **nodes** (`dict`, `list`, `tuple`) and transforms **leaves** (arrays). Model weights are usually a nested dict — no custom classes needed.

## 2-layer MLP as nested dict

`forward` takes `x` with a leading **batch** dimension `(B, D_in)` — same matmul + broadcast bias pattern from Episode 1. We do not use `vmap` here; the batch axis is part of the tensor shapes.

In [ ]:
def init_mlp(key, d_in, d_hidden, d_out):
    key, k0, k1 = jr.split(key, 3)
    return {
        0: {"w": jr.normal(k0, (d_in, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        1: {"w": jr.normal(k1, (d_hidden, d_out)) * 0.1, "b": jnp.zeros(d_out)},
    }


def forward(params, x):
    # x: (B, D_in) — bias (D_hidden,) broadcasts over batch
    x = jnp.tanh(x @ params[0]["w"] + params[0]["b"])
    x = x @ params[1]["w"] + params[1]["b"]
    return x


key = jr.key(0)
params = init_mlp(key, 8, 16, 4)
B = 32
key, k_x = jr.split(key)
x_batch = jr.normal(k_x, (B, 8))
print("forward out:", forward(params, x_batch).shape)

leaves = jax.tree.leaves(params)
print(f"{len(leaves)} leaves:", [leaf.shape for leaf in leaves])

## Inspect structure with `tree.map`

In [ ]:
shapes = jax.tree.map(lambda a: a.shape, params)
print(shapes)

## SGD — fixed learning rate, no optimizer state

In [ ]:
LEARNING_RATE = 0.01


def sgd_update(params, grads):
    return jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)


def loss_fn(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)


key, k_x, k_y = jr.split(jr.key(1), 3)
x_batch = jr.normal(k_x, (B, 8))
y_batch = jr.normal(k_y, (B, 4))
loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)
params = sgd_update(params, grads)
print("batched loss after one step:", loss)

## Training loop — return new params each step

In [ ]:
def train_step(params, x, y):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    params = sgd_update(params, grads)
    return params, loss


params = init_mlp(jr.key(3), 8, 16, 4)
losses = []
for _ in range(20):
    params, loss = train_step(params, x_batch, y_batch)
    losses.append(float(loss))
print("first loss:", losses[0])
print("last loss: ", losses[-1])

## `jit` the training step

In [ ]:
jitted_step = jax.jit(train_step)
params = init_mlp(jr.key(4), 8, 16, 4)
params, loss = jitted_step(params, x_batch, y_batch)
print("jitted step loss:", loss)

> **Key insight:** Return **new** params every step — never mutate in place. Batch with a leading dimension on `x` and `y`; `tree_map` applies the same SGD rule to every leaf.

---
## Exercise

1. Print all leaf shapes in the 2-layer MLP `params` dict.
2. Implement SGD with `tree_map` in two lines.
3. Run 20 batched steps on `(B, 8)` inputs; print the loss curve.

*(Solution below.)*

In [ ]:
print('final loss:', losses[-1])

**Next:** Part II — build a GPT-2 transformer in pure JAX.